[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CicadaUY/tokenpalooza/blob/main/recursos/prompting_finetuning_llm.ipynb)

Redes Neuronales para Lenguaje Natural, 2026

---
# **Grandes Modelos de Lenguaje (LLMs) para clasificación de emoción en tweets**

En este notebook haremos experimentos utilizando clasificando emociones en tweets usando un LLM mediante técnicas de prompting y fine-tuning.

**Importante:** Se recomienda ejecutar usando un runtime con GPU.

---



## Librerías, configuraciones y funciones auxiliares

Ejecutar las siguientes celdas para instalar las librerías necesarias, configurar formato de salida de Colab (word-wrap) y definir funciones necesarias más adelante para evaluación.

In [ ]:
# En algunos entornos Colab viene una versión antigua de torchao que interfiere con PEFT.
%pip uninstall -y torchao
%pip install -q -U 'transformers>=5.2,<6' 'accelerate>=1.1,<2' 'peft>=0.18,<1'

In [ ]:
import torch
import numpy as np

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EMOTIONS_ES = ["Alegría", "Tristeza", "Desagrado", "Ira", "Miedo", "Sorpresa", "Otras"]
EMOTIONS_ES_DICT = { e:i for (i,e) in enumerate(EMOTIONS_ES) }
EMOTIONS_EN = ["joy", "sadness", "disgust", "anger", "fear", "surprise", "others"]
EMOTIONS_EN_DICT = { e:i for (i,e) in enumerate(EMOTIONS_EN) }
EMOTIONS_ES_EN = { es:en for (es,en) in zip(EMOTIONS_ES,EMOTIONS_EN) }
EMOTIONS_EN_ES = { en:es for (en,es) in zip(EMOTIONS_EN,EMOTIONS_ES) }

In [ ]:
# Funciones auxiliares
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_recall_fscore_support

def plot_confusion_matrix(reference, prediction, labels):
    cm = confusion_matrix(reference, prediction, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot()
    plt.show()

def print_fscore(reference, prediction, labels):
    print("Macro:", f1_score(reference, prediction, average='macro'))
    scores = f1_score(reference, prediction, average=None, labels=labels)
    for i in range(len(labels)):
        print(f"Clase {labels[i]}:", + scores[i])


## Carga de la información de los tweets

Los datos que utilizaremos en este ejercicio son los de la competencia EmoEvalEs@IberLEF2021 (https://competitions.codalab.org/competitions/28682). En esta competencia se propone asignar la emoción principal de un tweet a una de 7 categorías:
* enojo / anger
* desagrado / disgust
* miedo / fear
* alegría / joy
* tristeza / sadness
* sorpresa / surprise
* otras / others

Descargaremos y utilizaremos las particiones de train y de dev, ya que la de test no cuenta con las etiquetas reales de la emoción esperada. Recordar que en un ejemplo real deberíamos partir el corpus de train para obtener una nueva partición de dev, o utilizar otra técnica como validación cruzada.

El corpus tiene más información, pero en este caso solo cargaremos cada ejemplo como <tweet,emoción>. Se imprime una entrada aleatoria de cada partición para probar que se hayan cargado correctamente.

In [ ]:
! wget https://www.fing.edu.uy/owncloud/index.php/s/cgbdFYX2te4zLrA/download/emoeval_data.zip
! unzip emoeval_data.zip

In [ ]:
import csv
import random

def load_corpus(corpusFile):
   with open(corpusFile, newline='') as corpus_csv:
      reader = csv.reader(corpus_csv, delimiter=',')
      titles = next(reader)
      corpus = [[x[2],EMOTIONS_EN_ES[x[4]]] for x in reader]
   return corpus

corpus_train = load_corpus('train.csv')
corpus_test = load_corpus('dev.csv')

print(random.choice(corpus_train))
print(random.choice(corpus_test))

Debido a que no contamos con las etiquetas reales de test, utilizaremos como partición de test la que originalmente se usó como dev en la competencia. Este conjunto que usaremos para evaluar las diferentes técnicas tiene **844 ejemplos**:

In [ ]:
print("Tamaño de corpus de test:", len(corpus_test))

Debido a que los grandes modelos de lenguaje requieren del uso de mucho cómputo, cada ejecución suele llevar una cantidad de tiempo no despreciable. Para que la evaluación lleve un tiempo razonable, utilizaremos **100 ejemplos aleatorios**.



In [ ]:
random.seed(123)
corpus_test = random.sample(corpus_test, 100)

print("Nuevo tamaño de corpus de test:", len(corpus_test))

for e in EMOTIONS_ES:
    print(e,len(list(t for t in corpus_test if t[1] == e)))

## Configuración de LLM

Utilizaremos una variante del modelo **Qwen 3.5** a través de la plataforma [HuggingFace](https://huggingface.co/).

Para cargar el LLM, son necesarios dos componentes: el **tokenizer**, encargado de dividir las secuencias de texto en partes más pequeñas llamadas tokens que el modelo es capaz de procesar, y el **modelo**, que es la red neuronal profunda (*transformer*) con 0.8 mil millones de parámetros.

Comenzaremos cargando el tokenizer ejecutando la siguiente celda.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")

A continuación veremos el proceso de tokenización realizado por el tokenizer para un tweet aleatorio del conjunto de train.

El resultado es una lista de identificadores, donde cada identificador corresponde a un token.

Complete la siguiente celda.

La documentación para trabajar con el tokenizer está disponible en: https://huggingface.co/docs/transformers/main_classes/tokenizer. En particular, se recomienda utilizar la función `convert_ids_to_tokens` para obtener los tokens a partir de la lista de ids.

In [ ]:
import random

tweet = random.choice(corpus_train)[0] # Elegir un tuit aleatorio
token_ids = tokenizer(tweet)['input_ids'] # Aplicar tokenizer al tuit
tokens = tokenizer.convert_ids_to_tokens(token_ids) # Obtener los tokens a partir de los ids

print("Tweet:", tweet)
print("\nTokens:", tokens)
print("\nIdentificadores de tokens:", token_ids)


A continuación, imprimimos el tamaño del vocabulario. Esto es, la cantidad total de tokens.

Para esto, se puede usar la función `get_vocab` del tokenizer, y `len` para obtener el tamaño.

In [ ]:
# Imprimir tamaño del vocabulario
print("Tamaño del vocabulario",len(tokenizer.get_vocab()))

A continuación cargaremos el modelo. La ejecución de la siguiente celda puede llevar algunos minutos, ya que se descargarán los parámetros del modelo y se cargaran en la memoria de la GPU.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    device_map="auto"
  )

La siguiente celda muestra la arquitectura del modelo.

In [ ]:
print(model)

Podemos observar que el modelo consiste de las siguientes partes:

- **embed_tokens:** La primer capa, encargada de asociar cada token a un embedding (vector de tamaño 1024).

- **Qwen3_5DecoderLayer:** Alterna entre una variante de self-attention llamada [Gated Deltanet](https://sebastianraschka.com/llms-from-scratch/ch04/08_deltanet/) y self-attention normal, en una relación 3:1.
- **lm_head**: Aplica una transformación lineal a la salida de la última capa, y retorna un vector de tamaño 248320.

## Generación de texto usando el LLM

En la siguiente celda se define la función que utilizaremos para generar texto con el LLM.

Se puede explorar los atributos de la clase GenerationConfig para ver más opciones que se pueden utilizar al momento de la generación:
https://huggingface.co/docs/transformers/en/main_classes/text_generation

In [ ]:
@torch.inference_mode()
def get_response(prompt, max_tok=96):
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    output = model.generate(
        **inputs,
        max_new_tokens=max_tok,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = output[0, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

Probamos la función anterior con una prompt de prueba. El modelo deberá completar la frase.

In [ ]:
prompt = "Uruguay está ubicado en"
output = get_response(prompt)

print(prompt + " " + output)

El modelo que estamos utilizando es un modelo de tipo "instruct". Esto significa que pasó por un proceso de fine-tuning para adaptarlo a las interacciones tipo chat.

Para esto, los modelos son entrenados con un formato de prompt específico (simulando un chat). Para obtener los mejores resultados es importante respetar este formato.

El tokenizer implementa la función `apply_chat_template`, que dado una lista de mensajes aplica el formato de prompt adecuado.

Cada mensaje de la lista es un diccionario que tiene un rol y el contenido del mensaje. Hay tres roles posibles:

- **system:** Este es un único mensaje que siempre debe ir primero. Se utiliza para dar instrucciones generales al modelo de lenguaje.

- **user:** Los mensajes de tipo usuario corresponden a los mensajes ingresados por el humano en una conversación con el modelo.

- **assistant:** Los mensajes de tipo asistente corresponden a los mensajes generados por el modelo de lenguaje.

El modelo de lenguaje es *stateless*. Esto significa que no mantiene el estado de la conversación. Es por eso que se debe mantener un diccionario con los mensajes intercambiados hasta el momento.

La siguiente función recibe una lista de mensajes y retorna un string que corresponde a la prompt, que luego será utilizada para generar una respuesta.

In [ ]:
def create_prompt(messages):
  return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

Veamos un ejemplo de prompt que sigue el formato adecuado, dada una lista de mensajes.

In [ ]:
messages = [
    {
      "role": "system",
      "content": "Eres un asistente servicial, respetuoso y honesto."
    },
    {
      "role": "user",
      "content": "Hola, ¿cómo estás?"
    },
    {
      "role": "assistant",
      "content": "Hola! Estoy dispuesto a ayudarte."
    },
    {
      "role": "user",
      "content": "¿Qué me puedes decir de Uruguay?"
    }
  ]

prompt = create_prompt(messages)
print(prompt)

In [ ]:
print(get_response(prompt))

## Experimento 1: Prompting con instrucciones

Este experimento consiste en **diseñar una prompt de sistema** adecuada para resolver la tarea. El objetivo es dar instrucciones al modelo del comportamiento esperado en el mensaje con rol de sistema. Luego, el tweet a clasificar será un mensaje de usuario, y se espera que el modelo retorne la etiqueta en el siguiente mensaje.

**Aspectos importantes a tener en cuenta:**

- Es importante dar el mayor contexto posible al modelo, explicando la tarea y las etiquetas. Recordar que el modelo no tiene conocimiento previo de la tarea que queremos que resuelva.
- El objetivo es que el primer token generado por el modelo sea el inicio de una de las siete etiquetas (Alegría, Tristeza, Desagrado, Ira, Miedo, Sorpresa, Otras), ya que luego generaremos un solo token. Es **fundamental** hacer esto explícito en la prompt, indicando que la respuesta esperada debe comenzar con la etiqueta.

In [ ]:
# Definición de prompt
def simple_prompt(tweet):
  messages = [
    {
      "role": "system",
      "content": """Eres un clasificador de emociones para tweets en español.
Tu tarea es leer el texto recibido en el próximo mensaje y clasificar la emoción principal en exactamente una de estas siete etiquetas:

- Alegría
- Tristeza
- Desagrado
- Ira
- Miedo
- Sorpresa
- Otras

### Objetivo de salida (muy importante)
Responde únicamente con una de las siete etiquetas exactamente como está escrita (sin espacios previos, sin comillas, sin emojis, sin puntuación, sin explicación).
No añadas texto extra ni saltos de línea adicionales."""
    },
    {
      "role": "user",
      "content": tweet
    }
  ]
  return create_prompt(messages)

A continuación veremos la prompt generada para algún tweet del corpus de train, y generamos la respuesta con el LLM para verificar que el comportamiento es el esperado.

In [ ]:
example = random.choice(corpus_train)
prompt = simple_prompt(example[0])
output = get_response(prompt, max_tok=20)

print("PROMPT:\n")
print(prompt)
print("\nRESPUESTA:\n")
print(output)
print("\nREFERENCIA:\n")
print(example[1])

Se puede ver que, a pesar de que podemos refinar el prompt, el modelo habitualmente dirá más palabras que las esperadas. Escribir un método que intente obtener la emoción predicha a partir de la respueta.

In [ ]:
def get_emotion_simple_prompt(tweet):
  prompt = simple_prompt(tweet)
  output = get_response(prompt, max_tok=20).lower()
  for e in EMOTIONS_ES:
    if e.lower() in output:
      return e
  return "Otras"

example = random.choice(corpus_train)
print(example[0])
print(get_emotion_simple_prompt(example[0]))
print(example[1])

Obtenemos métricas sobre todo el corpus de test para el caso zero-shot:

In [ ]:
reference = []
predictions = []
for t,e in corpus_test:
    p = get_emotion_simple_prompt(t)
    reference.append(e)
    predictions.append(p)

print_fscore(reference,predictions,EMOTIONS_ES)
plot_confusion_matrix(reference,predictions,EMOTIONS_ES)

Como lo que hace el modelo de lenguaje es devolvernos la probabilidad del siguiente token, podemos utilizar otra técnica para predecir la emoción: miramos el siguiente token predicho y vemos a cuál de las posibles emociones se le asigna más probabilidad.


In [ ]:
# Obtener el primer token de cada emoción
emotion_ids = tokenizer(EMOTIONS_ES, add_special_tokens=False).input_ids
emotion_tokens = [e[0] for e in emotion_ids]
emotion_tokens = torch.tensor(emotion_tokens)
print(list((e,tokenizer.decode(e)) for e in emotion_tokens))

def get_token_probabilities(prompt, model_llm):
    '''
    Dada una prompt, obtiene las probabilidades de los tokens de las emociones.

    Args:
        prompt (str): La prompt para el modelo de lenguaje.
        model_llm (AutoModelForCausalLM): El modelo de lenguaje.

    Returns:
        np.array: Un array con las probabilidades de los tokens de las emociones.
    '''
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model_llm(**inputs)

    # Obtener las probabilidades de los tokens
    next_token_logits = outputs.logits[:, -1, :]
    next_token_probs = torch.softmax(next_token_logits, dim=-1)

    # Obtener las probabilidades de los tokens de las emociones
    emotion_probs = next_token_probs[0, emotion_tokens].to(dtype=torch.float).cpu().numpy()

    # Normalizar las probabilidades
    emotion_probs /= emotion_probs.sum()

    return emotion_probs


El siguiente código nos permite obtener la emoción a partir de las probabilidades del primer token generado en la salida.



In [ ]:
import numpy as np

def get_emotion_first_token(tweet):
  prompt = simple_prompt(tweet)
  probs = get_token_probabilities(prompt, model)
  max_prob = np.argmax(probs)
  return EMOTIONS_ES[max_prob]

example = random.choice(corpus_train)
print(example[0])
print(get_emotion_first_token(example[0]))
print(example[1])


Obtenemos métricas sobre todo el corpus de test para el caso zero-shot con el primer token generado:

In [ ]:
reference = []
predictions = []
for t,e in corpus_test:
    p = get_emotion_first_token(t)
    reference.append(e)
    predictions.append(p)

print_fscore(reference,predictions,EMOTIONS_ES)

plot_confusion_matrix(reference,predictions,EMOTIONS_ES)

## Experimento 2: Few-Shot prompting

El experimento anterior es la manera más sencilla de utilizar el LLM, simplemente indicando la tarea a resolver. Una desventaja de este método es que no utiliza la información disponible en el conjunto de entrenamiento.

Una alternativa que suele mejorar los resultados es el método de **Few-Shot learning**. Few-Shot learning consiste en agregar algunos pocos ejemplos de la tarea resuelta en la prompt. Mediante los ejemplos (que se suman a las instrucciones), el modelo puede detectar patrones más fácilmente. Este método también se conoce como **In-Context Learning**, ya que, a diferencia de métodos como fine-tuning, se espera que el modelo aprenda a resolver una tarea nueva (para la que no fue entrenado) mediante ejemplos provistos en el contexto, sin modificaciones en sus parámetros.

El método utilizado en el experimento anterior se puede ver como el caso particular de Few-Shot donde $n = 0$. A este caso se le llama **Zero-Shot Learning**.

Para este experimento se debe completar la función `select_few_shot`, que debe retornar una lista con $n$ ejemplos elegidos con algún criterio del dataset pasado por parámetro.

Luego, estos $n$ ejemplos serán agregados a la lista de mensajes como (falsos) mensajes anteriores entre el usuario y el asistente.

**Aspectos importantes a tener en cuenta:**

- Se puede diseñar cualquier método para seleccionar los ejemplos. Algunos métodos posibles son:
  - Elegir los ejemplos de forma aleatoria.
  - Elegir los ejemplos de forma tal que haya la misma representación para cada clase.
  - Mirar el conjunto de entrenamiento y elegir a mano ejemplos que resulten característicos de la etiqueta que representan.
  - Etc.
- Nuevamente es necesario definir el prompt de sistema, al igual que en el experimento anterior. No es necesario diseñar una prompt de sistema nueva, se puede usar la misma que en experimento anterior.


In [ ]:
# Definición de prompt
import random

def select_few_shot(dataset, n=5):
  # Crear una lista con n ejemplos seleccionados del dataset, donde cada elemento es un diccionario con claves "tweet", "label"
  return random.sample(dataset, n)

def few_shot_prompt(tweet):
  messages = [
    {
      "role": "system",
      "content": """Eres un clasificador de emociones para tweets en español.
Tu tarea es leer el texto recibido en el próximo mensaje y clasificar la emoción principal en exactamente una de estas siete etiquetas:

- Alegría
- Tristeza
- Desagrado
- Ira
- Miedo
- Sorpresa
- Otras

### Objetivo de salida (muy importante)
Responde únicamente con una de las siete etiquetas exactamente como está escrita (sin espacios previos, sin comillas, sin emojis, sin puntuación, sin explicación).
No añadas texto extra ni saltos de línea adicionales."""
    }
  ]

  few_shot_examples = select_few_shot(corpus_train,10)
  for example in few_shot_examples:
    messages.append({"role": "user", "content": example[0]})
    messages.append({"role": "assistant", "content": example[1]})

  messages.append({"role": "user", "content": tweet})

  return create_prompt(messages)

A continuación veremos la prompt generada para el primer tweet del corpus de test, y generamos la respuesta con el LLM para verificar que el comportamiento es el esperado. Complete la siguiente celda.

In [ ]:
example = random.choice(corpus_train)
prompt = few_shot_prompt(example[0])
output = get_response(prompt, max_tok=20)

print("PROMPT:\n")
print(prompt)
print("\nRESPUESTA:\n")
print(output)
print("\nREFERENCIA:\n")
print(example[1])

Al igual que en el experimento 1, el modelo podrá generar palabras de más. Obtener la emoción predicha a partir de la respuesta.

In [ ]:
def get_emotion_few_shot(tweet):
  prompt = few_shot_prompt(tweet)
  output = get_response(prompt, max_tok=20).lower()
  for e in EMOTIONS_ES:
    if e.lower() in output:
      return e
  return "Otras"

example = random.choice(corpus_train)
print(example[0])
print(get_emotion_few_shot(example[0]))
print(example[1])

Obtenemos métricas sobre todo el corpus de test para el caso few-shot:

In [ ]:
reference = []
predictions = []
for t,e in corpus_test:
    p = get_emotion_few_shot(t)
    reference.append(e)
    predictions.append(p)

print_fscore(reference,predictions,EMOTIONS_ES)

plot_confusion_matrix(reference,predictions,EMOTIONS_ES)

Podemos volver a probar el método de obtener la emoción más probable del primer token, pero esta vez con few shot.

In [ ]:
import numpy as np

def get_emotion_first_token_few_shot(tweet):
  prompt = few_shot_prompt(tweet)
  probs = get_token_probabilities(prompt,model)
  max_prob = np.argmax(probs)
  return EMOTIONS_ES[max_prob]

example = random.choice(corpus_train)
print(example[0])
print(get_emotion_first_token_few_shot(example[0]))
print(example[1])


Obtenemos métricas sobre todo el corpus de test para el caso few-shot con el primer token generado:

In [ ]:
reference = []
predictions = []
for t,e in corpus_test:
    p = get_emotion_first_token_few_shot(t)
    reference.append(e)
    predictions.append(p)

print_fscore(reference,predictions,EMOTIONS_ES)

plot_confusion_matrix(reference,predictions,EMOTIONS_ES)

## Experimento 3: Modelo con fine-tuning

Los resultados hasta el momento han sido muy bajos, pero notar que son muchas categorías posibles, y estamos trabajando con modelos de tamaño muy pequeño que además no están ajustados para esta tarea en particular. ¿Qué tan bien funcionaría un modelo ajustado?

A continuación se provee el código para realizar un fine-tuning sobre el modelo que venimos usando (Qwen 3.5 0.8B). Debido a que realizar el fine-tuning **lleva ~30 minutos**, dejamos disponible el modelo ya finetuneado con los datos de train de EmoEvalEs en Hugging Face.

Se usa el mismo formato de conversación del modelo instruct, con un sistem prompt específico: _"Eres un clasificador de emociones para tweets en español. Las emociones posibles son: Alegría, Tristeza, Desagrado, Ira, Miedo, Sorpresa, Otras"_.

### Construcción del dataset para entrenar

In [ ]:
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

class EmotionsSFTDataset(Dataset):
    def __init__(self, corpus, tokenizer, max_length=512):
        self.rows = []
        self.lengths = []
        for tweet, emotion in corpus:
            prompt_messages = [
                {'role': 'system', 'content': 'Eres un clasificador de emociones para tweets en español. Las emociones posibles son: Alegría, Tristeza, Desagrado, Ira, Miedo, Sorpresa, Otras'},
                {'role': 'user', 'content': tweet},
            ]
            complete_messages = prompt_messages + [{'role': 'assistant', 'content': emotion}]

            prefix = tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=True,
                add_generation_prompt=True,
                enable_thinking=False,
            )['input_ids']

            full = tokenizer.apply_chat_template(
                complete_messages,
                tokenize=True,
                add_generation_prompt=False,
                enable_thinking=False,
            )['input_ids']

            labels = [-100] * len(prefix) + full[len(prefix):]

            self.rows.append({'input_ids': full, 'labels': labels})
            self.lengths.append(len(full))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        return {key: torch.tensor(value, dtype=torch.long) for key, value in row.items()}


def collate_sft(batch):
    # Construcción del batch: padding para entradas, etiquetas y attention_mask.
    ids = [example['input_ids'] for example in batch]
    labels = [example['labels'] for example in batch]
    return {
        'input_ids': pad_sequence(ids, batch_first=True, padding_value=tokenizer.pad_token_id),
        'attention_mask': pad_sequence([torch.ones_like(x) for x in ids], batch_first=True, padding_value=0),
        'labels': pad_sequence(labels, batch_first=True, padding_value=-100),
    }

train_sft = EmotionsSFTDataset(corpus_train, tokenizer, max_length=512)
print('Longitud máxima del entrenamiento:', max(train_sft.lengths), 'tokens')
print('Primer ejemplo, tokens a predecir:')
primer_ejemplo = train_sft[0]
print(tokenizer.decode(primer_ejemplo['input_ids'][primer_ejemplo['labels'] != -100]))

### Fine-tuning LoRA

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, TrainingArguments

NUM_EPOCHS = 1
LORA_R = 8
LORA_ALPHA = 16
BATCH_SIZE = 4
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4

# Qwen3.5 mezcla capas de atención completa y lineal.
# all-linear evita depender de los nombres particulares de cada proyección.
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    target_modules='all-linear',
    task_type=TaskType.CAUSAL_LM,
)

model.config.use_cache = False

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
train_args = TrainingArguments(
    output_dir='output_lora',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='linear',
    warmup_steps=0.05,
    optim='adamw_torch',
    logging_steps=10,
    save_strategy='no',
    eval_strategy='no',
    report_to='none',
    remove_unused_columns=False
)
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_sft,
    data_collator=collate_sft,
)
train_result = trainer.train()
print('Loss final reportado:', train_result.training_loss)

# Guardar solamente los adaptadores LoRA (sin duplicar todo el modelo base).
model.save_pretrained('adapter_lora')

In [ ]:
# Subir a Hugging Face

# Fusionar los pesos del LoRA con el modelo base
merged_model = trainer.model.merge_and_unload(safe_merge=True)

# Subir el modelo completo
repo_id = "nsuruguay05/Qwen3.5-0.8B-EmoEvalES"

merged_model.push_to_hub(repo_id)

# 4. Subir el tokenizer
tokenizer.push_to_hub(repo_id)

### Correr modelo

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer_ft = AutoTokenizer.from_pretrained("nsuruguay05/Qwen3.5-0.8B-EmoEvalES")
model_ft = AutoModelForCausalLM.from_pretrained("nsuruguay05/Qwen3.5-0.8B-EmoEvalES", device_map="auto")

In [ ]:
import numpy as np

def get_emotion_finetuned_first_token(tweet):
  prompt_messages = [
    {'role': 'system', 'content': 'Eres un clasificador de emociones para tweets en español. Las emociones posibles son: Alegría, Tristeza, Desagrado, Ira, Miedo, Sorpresa, Otras'},
    {'role': 'user', 'content': tweet},
  ]
  prompt = tokenizer_ft.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
  )
  probs = get_token_probabilities(prompt, model_ft)
  max_prob = np.argmax(probs)
  return EMOTIONS_ES[max_prob]

example = random.choice(corpus_train)
print(example[0])
print(get_emotion_finetuned_first_token(example[0]))
print(example[1])

Obtenemos métricas sobre todo el corpus de test para el caso finetuneado con el primer token generado:

In [ ]:
reference = []
predictions = []
for t,e in corpus_test:
    p = get_emotion_finetuned_first_token(t)
    reference.append(e)
    predictions.append(p)

print_fscore(reference,predictions,EMOTIONS_ES)

plot_confusion_matrix(reference,predictions,EMOTIONS_ES)